### 1. Install Dependencies

In [19]:
from huggingface_hub import notebook_login
notebook_login()
from datasets import load_dataset

### 2. Load Dataset

In [48]:
dataset_id = 'AI-MO/NuminaMath-TIR'
train_dataset, test_dataset = load_dataset(dataset_id, split=['train[:2%]', 'test[:2%]'])

print(train_dataset)
print(train_dataset[0])
print("PROBLEM ", train_dataset[0]["problem"])
print("SOLUTION ", train_dataset[0]["solution"])
print("MESSAGES ", train_dataset[0]["messages"])


Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 1449
})
{'problem': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'solution': "To determine the coefficient of \\(x^2y^6\\) in the expansion of \\(\\left(\\frac{3}{5}x - \\frac{y}{2}\\right)^8\\), we can use the binomial theorem.\n\nThe binomial theorem states:\n\\[\n(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} b^k\n\\]\n\nIn this case, \\(a = \\frac{3}{5}x\\), \\(b = -\\frac{y}{2}\\), and \\(n = 8\\).\n\nWe are interested in the term that contains \\(x^2y^6\\). In the general term of the binomial expansion:\n\\[\n\\binom{8}{k} \\left(\\frac{3}{5}x\\right)^{8-k} \\left(-\\frac{y}{2}\\right)^k\n\\]\n\nTo get \\(x^2\\), we need \\(8 - k = 2\\), thus \\(k = 6\\).\n\nSubstituting \\(k = 6\\) into the expression:\n\\[\n\\binom{8}{6} \\left(\\frac{3}{5}x\\right)^{8-6} \\left(-\\frac{y}{2}\\right)^6 = \\binom{8}{

In [49]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

train_dataset = train_dataset.map(make_conversation)
test_dataset = test_dataset.map(make_conversation)

Map:   0%|          | 0/1449 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [50]:
print(train_dataset[0]['prompt'])

[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer>', 'role': 'system'}, {'content': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'role': 'user'}]


In [51]:
train_dataset = train_dataset.remove_columns(['messages', 'problem'])
print(train_dataset)

Dataset({
    features: ['solution', 'prompt'],
    num_rows: 1449
})


### 3. Post-Training the Base Model Using GRPO

In [52]:
import torch
from transformers import AutoModelForCausalLM

model_id = "Qwen/Qwen2-0.5B-Instruct"
#model = AutoModelForCausalLM.from_pretrained(
#    model_id,
#    torch_dtype="auto",
#    device_map="auto",
#)

print("cuda available: ", torch.cuda.is_available())

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
)

cuda available:  True


In [53]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [54]:
import re
def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content) for content in completion_contents]
    rewards_list = [1.0 if match else 0.0 for match in matches]
    return [1.0 if match else 0.0 for match in matches]

In [55]:
# I honestly don't understand how content and solution are compared

from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions, **kwargs):
    """Reward function that checks if the completion is the same as the ground truth."""
    solutions = kwargs['solution']
    completion_contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, solution in zip(completion_contents, solutions):
        gold_parsed = parse(solution, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        answer_parsed = parse(content, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        if len(gold_parsed) != 0:
            try:
                rewards.append(float(verify(answer_parsed, gold_parsed)))
            except Exception:
                rewards.append(0.0)
        else:
            rewards.append(1.0)
    return rewards

In [56]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    output_dir="Qwen2-0.5B-GRPO-test",
    learning_rate=1e-5,
    remove_unused_columns=False, # to access the solution column in accuracy_reward
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    #bf16=True,
    bf16=False,
    fp16=True,

    # Parameters that control de data preprocessing
    max_completion_length=64, # default: 256
    num_generations=4, # default: 8
    max_prompt_length=128, # default: 512

    # Parameters related to reporting and saving
    report_to=["tensorboard"],
    logging_steps=10,
    push_to_hub=True,
    save_strategy="steps",
    save_steps=10,
)

In [57]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset
)

In [58]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="equations=True in NormalizationConfig is deprecated, as it handled by the parser now",
    category=UserWarning,
)

In [59]:
trainer.train()

equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handle

Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000100
50,0.000100
60,0.000100
70,0.000200
80,0.000200
90,0.000200


equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handled by the parser now
equations=True in NormalizationConfig is deprecated, as it handle

TrainOutput(global_step=90, training_loss=0.00010305306938486561, metrics={'train_runtime': 1088.447, 'train_samples_per_second': 1.331, 'train_steps_per_second': 0.083, 'total_flos': 0.0, 'train_loss': 0.00010305306938486561})

In [60]:
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(dataset_name=dataset_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/s77estef/Qwen2-0.5B-GRPO-test/commit/1dd64e3d1ba90872f02f62a06657b1e6e422600b', commit_message='End of training', commit_description='', oid='1dd64e3d1ba90872f02f62a06657b1e6e422600b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/s77estef/Qwen2-0.5B-GRPO-test', endpoint='https://huggingface.co', repo_type='model', repo_id='s77estef/Qwen2-0.5B-GRPO-test'), pr_revision=None, pr_num=None)

### 4. Check Model Performance

In [61]:
from transformers import AutoTokenizer

model_id = "sergiopaniego/Qwen2-0.5B-GRPO"
trained_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)
trained_tokenizer = AutoTokenizer.from_pretrained(model_id)

adapter_config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

In [62]:
print(test_dataset['prompt'][0])

[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer>', 'role': 'system'}, {'content': "In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?", 'role': 'user'}]


In [63]:
import time

def generate_with_reasoning(prompt):
  # Build the prompt from the dataset
  prompt = " ".join(entry['content'] for entry in prompt)

  # Tokenize and move to the same device as the model
  inputs = trained_tokenizer(prompt, return_tensors="pt").to(trained_model.device)

  # Generate text without gradients
  start_time = time.time()
  with torch.no_grad():
      output_ids = trained_model.generate(**inputs, max_length=500)
  end_time = time.time()

  # Decode and extract model response
  generated_text = trained_tokenizer.decode(output_ids[0], skip_special_tokens=True)

  # Get inference time
  inference_duration = end_time - start_time

  # Get number of generated tokens
  num_input_tokens = inputs['input_ids'].shape[1]
  num_generated_tokens = output_ids.shape[1] - num_input_tokens

  return generated_text, inference_duration, num_generated_tokens

In [64]:
prompt = test_dataset['prompt'][0]
generated_text, inference_duration, num_generated_tokens = generate_with_reasoning(prompt)
print(generated_text)

A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer> In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person? <think> The sum of the digits of their birth year is 4.</think> <answer>20</answer>

The sum of the digits of their birth year is 4. This means that the person must have been born on a leap year (in which case the sum would be 366). Since 1988 was not a leap year, we can conclude that the person must have been born on a normal year. Let's check if 1988 is a leap year.

<think> Reasoning process here </think> The sum of the digits of their birth year is 4. Since 1988 is not a leap year, the su

In [65]:
print(f"Inference time: {inference_duration:.2f} seconds")
print(f"Generated tokens: {num_generated_tokens}")

Inference time: 3.73 seconds
Generated tokens: 387


In [66]:
prompt_text = " ".join(entry['content'] for entry in prompt)
response_text = generated_text[len(prompt_text):].strip()
print(response_text)

<think> The sum of the digits of their birth year is 4.</think> <answer>20</answer>

The sum of the digits of their birth year is 4. This means that the person must have been born on a leap year (in which case the sum would be 366). Since 1988 was not a leap year, we can conclude that the person must have been born on a normal year. Let's check if 1988 is a leap year.

<think> Reasoning process here </think> The sum of the digits of their birth year is 4. Since 1988 is not a leap year, the sum would be less than 366. Therefore, the person must have been born on a normal year. 

Since 1988 is not a leap year, we can conclude that the person must have been born on a normal year. Let's check if 1988 is a leap year.
<think> Reasoning process here </think> The sum of the digits of their birth year is 4. Since 1988 is not a leap year, the sum would be less than 366. Therefore, the person must have been born on a normal year. 

Since 1988 is not a leap year, we can conclude that the person mu